# Reproduce Paper Figures and Tables

This notebook generates all figures and tables from the paper:
**"Machine Learning for Aquatic Ecotoxicity Prediction with Quantified Uncertainty"**

### Prerequisites
1. Install dependencies: `pip install -r requirements.txt`
2. Place ADORE data in `data/raw/` (see `data/raw/README.md`)
3. Train the model: `python scripts/train_bfm.py`
4. Generate predictions: `python scripts/generate_predictions.py`

Once the trained model artifacts exist in `outputs/models/`, this notebook reproduces every figure and table.

## Setup

In [1]:
print('lezcz')

lezcz


In [2]:
import sys
from pathlib import Path

# Project root (one level up from notebooks/)
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR / "src"))
sys.path.insert(0, str(ROOT_DIR / "analysis"))

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

## Configuration

Adjust these parameters to tweak the analysis. Changes here propagate to all figures below.

In [3]:
# Duration to filter to for most analyses
DURATION_HOURS = 48

# Chemicals for SSD case studies (Figures 7-9)
SSD_CHEMICALS = {
    "Atrazine": "1912-24-9",
    "Chlorfenprop-methyl": "14437-17-3",
}

# HC percentile for Figures 10-11
HC_PERCENTILE = 20

# Uncertainty example plots (Figure 5)
N_EXAMPLE_CHEMICALS = 5
MIN_OBS_FOR_EXAMPLES = 10
RANDOM_SEED = 42

# MCMC settings for SSD uncertainty (Figure 7)
N_SSD_CURVES = 2000

# Figure output settings
SAVE_FIGURES = True  # Set to False to just display without saving
FIGURE_DPI = 150

## Load Data

Load the ADORE dataset and model outputs once; reused by all figures.

In [ ]:
import numpy as np
import pandas as pd
from data.load_ecotox import load_ecotox_data

DATA_DIR = ROOT_DIR / "data" / "raw"
MODELS_DIR = ROOT_DIR / "outputs" / "models"

# Load raw dataset
full_data, y_centered, y_mean = load_ecotox_data(
    adore_path=DATA_DIR / "ecotox_mortality_processed.csv",
    chemicals_path=DATA_DIR / "ecotox_properties_with-oecd-function.csv",
    use_molar=False,
    use_selfies=False, use_mol2vec=False, use_fingerprint=False,
    shuffle=True, random_state=42,
)
full_data["y_true"] = y_centered + y_mean

print(f"Loaded {len(full_data):,} observations")
print(f"  Unique chemicals: {full_data['CAS'].nunique():,}")
print(f"  Unique species:   {full_data['species'].nunique():,}")
print(f"  Centering mean:   {y_mean:.4f} log mg/L")

In [ ]:
# Load out-of-fold predictions (from cross-validation)
oof_mean = np.load(MODELS_DIR / "oof_mean.npy")
oof_epistemic = np.load(MODELS_DIR / "oof_epistemic.npy")
oof_aleatoric = np.load(MODELS_DIR / "oof_aleatoric.npy")

df = full_data.copy()
df["y_pred"] = oof_mean + y_mean
df["epistemic_var"] = oof_epistemic
df["aleatoric_var"] = oof_aleatoric
df["total_var"] = df["epistemic_var"] + df["aleatoric_var"]
df["epistemic_sd"] = np.sqrt(df["epistemic_var"])
df["aleatoric_sd"] = np.sqrt(df["aleatoric_var"])
df["total_sd"] = np.sqrt(df["total_var"])

print("Loaded OOF predictions.")

In [6]:
# Load full prediction matrix (from full-dataset training)
pred_df = pd.read_parquet(MODELS_DIR / "full_predictions.parquet")
pred_df["epistemic_sd"] = np.sqrt(pred_df["pred_epistemic_var"])
pred_df["aleatoric_sd"] = np.sqrt(pred_df["pred_aleatoric_var"])

print(f"Loaded {len(pred_df):,} full predictions.")

Loaded 16,699,060 full predictions.


---
## Dataset Characterization

### Table 1: Summary Statistics

In [7]:
from dataset_figures import table1_summary

table1_summary(full_data)


TABLE 1: Summary statistics of the ADORE dataset
  Number of unique chemicals                                   3295
  Number of unique species                                     1267
  Number of durations                                          4
  Number of unique (species, chemical, duration) triplets observed 31319
  Total number of observations                                 70670
  Sparsity (species*chemical matrix)                           99.2%
Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/dataset/table1_summary.csv


### Figure 1: Rank-Frequency Plots

In [8]:
from dataset_figures import figure1_rank_frequency

figure1_rank_frequency(full_data)

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/dataset/rank_frequency_plots.png


### Figure 2: Distribution of RSDs

In [9]:
from dataset_figures import figure2_rsd_distribution

figure2_rsd_distribution(full_data)

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/dataset/rsd_distribution.png


---
## Model Performance

### Figure 3: Predicted vs Measured Correlation

In [10]:
from analyze_results import plot_predicted_vs_measured_correlation

corr_stats = plot_predicted_vs_measured_correlation(df, duration_hours=DURATION_HOURS)


PREDICTED vs MEASURED TOXICITY CORRELATION (48h)
Observations at 48h: 19,520
Aggregated to 8,845 unique chemical/species pairs
   Unique chemicals: 2,382
   Unique species: 675

Correlation Statistics:
   Pearson r:  0.8832
   R²:         0.7801
   p-value:    0.00e+00
   RMSE:       0.7567
   MAE:        0.4940

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/analyze_results/predicted_vs_measured_48h.png


### Figure 4: Residual Bias Analysis

In [11]:
from analyze_results import analyze_prediction_bias

bias_stats = analyze_prediction_bias(df, duration_hours=DURATION_HOURS)


SYSTEMATIC BIAS ANALYSIS (48h)

1. OVERALL BIAS STATISTICS
Mean Bias (Pred - Meas):     -0.0109
Median Bias:                 -0.0216
Std Dev of Residuals:        0.7566

T-test (H0: mean bias = 0):
   t-statistic:              -1.3574
   p-value:                  1.75e-01
   → No significant systematic bias detected

Overpredictions:  4,189 (47.4%)
Underpredictions: 4,656 (52.6%)

2. BIAS BY TOXICITY LEVEL

Toxicity Level          Mean Bias      Std Dev      Count
--------------------------------------------------------
Very Low (<-4)            +1.2978       1.5885         62
Low (-4 to -2)            +0.6557       0.9264        757
Medium (-2 to 0)          +0.2511       0.6593       2756
High (0 to 2)             -0.1521       0.5267       4162
Very High (>2)            -0.6613       0.8095       1108

3. RESIDUAL DISTRIBUTION
Skewness:  +0.4951 (approximately symmetric)
Kurtosis:  +6.8018 (heavy tails: more outliers than normal)

Residual Percentiles:
    1th percentile: -2.1049
 

---
## Uncertainty Calibration

### Table 2: Aleatoric Calibration by Replicate Count

In [12]:
from dataset_figures import table2_aleatoric_calibration

table2_aleatoric_calibration(full_data)


TABLE 2: Aleatoric calibration by replicate count
Replicates   N groups   Mean Ratio   Median Ratio   Mean Pred SD   Mean Emp SD 
-------------------------------------------------------------------------------
5-9          1805       1229986361056.385 1.754          0.566          0.387       
10-19        658        2.534        1.635          0.521          0.381       
20-49        227        1.877        1.436          0.516          0.400       
50-99        30         1.532        1.153          0.537          0.463       
100+         6          1.014        0.968          0.537          0.558       
Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/dataset/table2_aleatoric_calibration.csv


### Figure 5: Example Predictions with Uncertainty

In [13]:
from uncertainty_figures import plot_example_predictions

plot_example_predictions(df)


FIGURE 2: Example Predictions with Uncertainty
Chemicals with ≥10 observations at 48h: 347

Selected chemicals:
  2,6-Dimethylquinoline (CAS 877-43-0): 10 observations
  Alachlor (CAS 15972-60-8): 23 observations
  Picric acid (CAS 88-89-1): 12 observations
  Sodium bromide (CAS 7647-15-6): 20 observations
  Bisphenol A (CAS 80-05-7): 21 observations

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/uncertainty_calibration/example_predictions_48h.png


### Figure 6: Uncertainty vs Data Availability

In [14]:
from uncertainty_figures import plot_uncertainty_vs_observations

chem_stats = plot_uncertainty_vs_observations(df)


FIGURE 1: Uncertainty vs Number of Observations (OOF)
Chemicals at 48h: 2382
Excluded 660 cold-start chemicals (< 3 total obs)
Remaining: 1722
Total observation range: 3 - 2651

Spearman correlations:
  Epistemic SD vs # obs: ρ = -0.649 (p = 1.95e-206)
  Aleatoric SD vs # obs: ρ = -0.824 (p = 0.00e+00)

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/uncertainty_calibration/uncertainty_vs_observations_48h.png


---
## Species Sensitivity Distributions (SSDs)

### Figures 7-9: SSD Analysis per Chemical

For each chemical in `SSD_CHEMICALS`:
- **Figure 7**: Ensemble SSDs from posterior samples (epistemic uncertainty band)
- **Figure 8**: Novel SSD with per-species uncertainty bars
- **Figure 9**: Traditional vs novel SSD overlay

In [15]:
import ssd_analysis
from ssd_analysis import (
    set_target, filter_observations, filter_predictions,
    plot_traditional_ssd, plot_novel_ssd_with_uncertainty,
    plot_traditional_vs_novel_ssd,
)
from ssd_mc_uncertainty import plot_ssd_with_uncertainty

for chem_name, cas in SSD_CHEMICALS.items():
    print("\n" + "=" * 70)
    print(f"  {chem_name} (CAS {cas})")
    print("=" * 70)

    set_target(cas, chem_name)

    # Filter observations and predictions to this chemical
    df_obs_filtered = filter_observations(full_data)
    pred_filtered = filter_predictions(pred_df)

    # Figure 7: MCMC posterior ensemble SSD
    plot_ssd_with_uncertainty(pred_filtered, n_curves=N_SSD_CURVES)

    # Figure 8: Novel SSD with uncertainty bars
    plot_novel_ssd_with_uncertainty(pred_filtered)

    # Figure 9: Traditional vs novel comparison
    plot_traditional_vs_novel_ssd(df_obs_filtered, pred_filtered)


  Atrazine (CAS 1912-24-9)

Filtered observations to Atrazine (CAS 1912-24-9) at 48h:
   Observations: 143
   Unique species: 54

Filtered predictions to Atrazine at 48h:
   Predictions for 1267 species

SSD WITH MCMC POSTERIOR SAMPLES (2000 curves)
Loading trained model from /home/tad/HierarchicalBFMEcotox/outputs/models/trained_model.pkl...


/home/tad/HierarchicalBFMEcotox/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tad/HierarchicalBFMEcotox/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tad/HierarchicalBFMEcotox/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:

Model has 1950 posterior samples
Built design matrix: (1267, 4879)
Chemical group index: 50
Computing predictions for 1950 posterior samples...
Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/ssd_uncertainty_atrazine_48h.png

HC5 RESULTS (5% of species affected)

Metric               Log mg/L        mg/L           
--------------------------------------------------
HC5 Median           -1.486          0.226342       
HC5 Lower (2.5%)     -1.636          0.194726       
HC5 Upper (97.5%)    -1.354          0.258218       
--------------------------------------------------
95% CI Width         0.282          

NOVEL SSD WITH UNCERTAINTY BARS
Using 1267 species with uncertainty estimates
Mean prediction range: [-2.75, 3.06]
Aleatoric SD (constant): 0.401
Epistemic SD range: [0.059, 1.155]
Total SD range: [0.406, 1.223]
Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/ssd_novel_uncertainty_atrazine_48h.png

TRADITIONAL vs NOVEL SSD (same panel)
Tradit

/home/tad/HierarchicalBFMEcotox/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tad/HierarchicalBFMEcotox/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tad/HierarchicalBFMEcotox/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler fr

Model has 1950 posterior samples
Built design matrix: (1267, 4879)
Chemical group index: 1297
Computing predictions for 1950 posterior samples...
Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/ssd_uncertainty_chlorfenprop_methyl_48h.png

HC5 RESULTS (5% of species affected)

Metric               Log mg/L        mg/L           
--------------------------------------------------
HC5 Median           -2.729          0.065310       
HC5 Lower (2.5%)     -3.524          0.029475       
HC5 Upper (97.5%)    -1.800          0.165302       
--------------------------------------------------
95% CI Width         1.724          

NOVEL SSD WITH UNCERTAINTY BARS
Using 1267 species with uncertainty estimates
Mean prediction range: [-3.50, 0.85]
Aleatoric SD (constant): 1.266
Epistemic SD range: [0.466, 1.247]
Total SD range: [1.350, 1.777]
Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/ssd_novel_uncertainty_chlorfenprop_methyl_48h.png

TRADITIONAL vs NOVEL

---
## HC20 Comparison (Figures 10-11)

This section computes HC20 values for all chemicals using MCMC posterior samples, then compares them with traditional lognormal-fitted HC20 values.

**Note:** This cell is computationally expensive (iterates over all chemicals x all posterior samples). It saves results to CSV so subsequent runs of the plotting cells can skip recomputation.

In [16]:
from ssd_mc_uncertainty import compute_hcx_all_chemicals, compute_traditional_hcx

hcx_csv_path = ROOT_DIR / "outputs" / "figures" / "ssd_analysis" / f"hcx_comparison_{DURATION_HOURS}h.csv"

if hcx_csv_path.exists():
    print(f"Loading cached HCx comparison from {hcx_csv_path}")
    df_hcx = pd.read_csv(hcx_csv_path)
else:
    print("Computing HCx for all chemicals (this may take a while)...")
    df_mc = compute_hcx_all_chemicals(percentiles=[HC_PERCENTILE])

    df_trad = compute_traditional_hcx(full_data, percentiles=[HC_PERCENTILE])

    df_hcx = df_mc.merge(df_trad, on="CAS", how="left")
    df_hcx.to_csv(hcx_csv_path, index=False)
    print(f"Saved to {hcx_csv_path}")

print(f"HCx data: {len(df_hcx)} chemicals")

Loading cached HCx comparison from /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/hcx_comparison_48h.csv
HCx data: 3295 chemicals


### Figure 10: HC20 Forest Plot

In [17]:
from hcx_plots import figure10_forest_plot

figure10_forest_plot(df_hcx)

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/hc20_forest_plot.png


### Figure 11: HC20 Correlation (Traditional vs BFM)

In [ ]:
from hcx_plots import figure11_correlation_scatter

figure11_correlation_scatter(df_hcx)

Saved: /home/tad/HierarchicalBFMEcotox/outputs/figures/ssd_analysis/hc20_correlation_trad_vs_mc.png
  Pearson r = 0.950
  N chemicals = 412


: 

---
## Summary

All figures and tables have been generated. Outputs are saved to `outputs/figures/`.